# Rejection sampling + SFT: model tự học từ lời giải đúng của chính nó

Cách rẻ nhất để có dữ liệu reasoning là để model tự sinh rồi **chỉ giữ lời giải kiểm được là đúng** (rejection sampling,
STaR, RFT). Notebook này làm trọn vòng đó với `Qwen2.5-0.5B-Instruct` trên GSM8K: sinh 4 lời giải cho mỗi câu train, lọc
bằng reward của bài Basic 01, khử trùng lặp, kiểm nhiễm bẩn với tập đo, SFT bằng LoRA (TRL, như module 14), rồi đo
trước/sau trên 500 câu test ghim. Bài Advanced 02 dùng lại đúng khung này với model thầy lớn hơn.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng, hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) tải model, sinh, train, đo; test bỏ qua chúng.
- Chạy thật bằng kernel `ai-gpu` (GPU 4GB là đủ). Kernel CPU chỉ chạy bản thu nhỏ để kiểm code chạy hết.
- Chấm: `uv run pytest tracks/ai-engineer/modules/15-reasoning-models/exercises/intermediate/03-rejection-sampling-sft -q`

In [ ]:
# explore: tìm gốc repo, tải GSM8K nếu chưa có
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


def dataset_dir(name):
    try:
        from data.download_dataset import ensure
    except ImportError:
        path = ROOT / "data" / "raw" / name
        if not path.is_dir():
            raise RuntimeError(f"Chưa có dữ liệu: uv run python data/download_dataset.py {name}")
        return path
    return ensure(name)


GSM = dataset_dir("gsm8k")
CACHE = GSM / "cache"
CACHE.mkdir(exist_ok=True)
print(GSM)

In [ ]:
import re
from fractions import Fraction

## 1. Dán reward của Basic 01

Dán toàn bộ phần thân `answers.py` (regex, `normalize_number`, `extract_answer`, ba hàm reward).

In [ ]:
_THOUSANDS = re.compile(r"(?<=\d),(?=\d{3}(?!\d))")
_PLAIN = re.compile(r"[+-]?(?:\d+/\d+|\d+(?:\.\d+)?)")
NUMBER = re.compile(r"(?<![\w.])[-−]?\$?\d{1,3}(?:,\d{3})+(?:\.\d+)?(?!\d)|(?<![\w.])[-−]?\$?(?:\d+/\d+|\d+(?:\.\d+)?)")
_ANSWER_TAG = re.compile(r"<answer>(.*?)</answer>", re.S)
_BOXED = re.compile(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}")
_HASHES = re.compile(r"####(.*)")
_PHRASE = re.compile(r"answer\s*(?:is|:)(.*)", re.I)
_LETTER = re.compile(r"[^\W\d_]")
_WORD = re.compile(r"[^\W_]")


def normalize_number(text: str) -> str | None:
    """Dạng chuẩn của một chuỗi chỉ chứa MỘT số, hoặc None nếu không phải đúng một số.

    Dạng chuẩn là str(Fraction(...)): "18", "-3", "5/2". Chấp nhận khoảng trắng hai đầu, dấu "$" và "%",
    dấu phẩy ngăn hàng nghìn ("1,234"), dấu chấm cuối câu ("18."), dấu "+" đầu, dấu trừ unicode "−",
    số thập phân ("2.50") và phân số ("3/4"). Mọi thứ khác (chữ, hai số, "1,2") cho None.
    """
    s = text.strip().replace("−", "-").replace("\\$", "").replace("$", "").replace("%", "").strip()
    s = _THOUSANDS.sub("", s)
    if s.endswith("."):
        s = s[:-1]
    if not _PLAIN.fullmatch(s):
        return None
    try:
        return str(Fraction(s))
    except ZeroDivisionError:
        return None


def gold_answer(answer_field: str) -> str:
    """Đáp số chuẩn hoá của cột `answer` GSM8K (phần sau "####")."""
    value = normalize_number(answer_field.split("####")[-1])
    if value is None:
        raise ValueError(f"không đọc được đáp số: {answer_field[-40:]!r}")
    return value


def _numbers(text: str) -> list[str]:
    found = (normalize_number(m.group(0)) for m in NUMBER.finditer(text))
    return [n for n in found if n is not None]


def extract_answer(completion: str) -> str | None:
    """Đáp số của một câu trả lời tự do, theo thứ tự ưu tiên ở README của bài."""
    for pattern, pick in ((_ANSWER_TAG, -1), (_BOXED, -1)):
        spans = pattern.findall(completion)
        if spans:
            numbers = _numbers(spans[-1])
            if numbers:
                return numbers[pick]
    for pattern in (_HASHES, _PHRASE):
        spans = pattern.findall(completion)
        if spans:
            numbers = _numbers(spans[-1])
            if numbers:
                return numbers[0]
    numbers = _numbers(completion)
    return numbers[-1] if numbers else None


def is_correct(completion: str, gold: str) -> bool:
    """extract_answer(completion) bằng đáp số gold (đã chuẩn hoá)."""
    return extract_answer(completion) == gold


def format_reward(completion: str) -> float:
    """1.0 nếu có lập luận rồi đúng một \\boxed{SỐ} ở cuối (sau đó không còn chữ hay số), ngược lại 0.0."""
    if completion.count("\\boxed{") != 1:
        return 0.0
    match = _BOXED.search(completion)
    if not match or normalize_number(match.group(1)) is None:
        return 0.0
    before, after = completion[:match.start()], completion[match.end():]
    return 1.0 if _LETTER.search(before) and not _WORD.search(after) else 0.0


def correctness_reward(completion: str, gold: str) -> float:
    """1.0 nếu \\boxed{} cuối cùng chứa đúng một số bằng gold, ngược lại 0.0."""
    spans = _BOXED.findall(completion)
    return 1.0 if spans and normalize_number(spans[-1]) == gold else 0.0


def total_reward(completion: str, gold: str, format_weight: float = 0.2) -> float:
    """correctness_reward + format_weight * format_reward."""
    return correctness_reward(completion, gold) + format_weight * format_reward(completion)

## 2. Lọc lời giải, kiểm nhiễm bẩn, dựng mẫu SFT

- `select_traces(samples, gold, max_keep=2) -> list[str]`: giữ lời giải có **cả** `correctness_reward` và
  `format_reward` bằng 1; bỏ trùng lặp sau khi gộp mọi khoảng trắng liên tiếp thành một dấu cách và `strip` (giữ bản
  xuất hiện đầu); xếp theo độ dài ký tự tăng dần (bằng nhau thì giữ thứ tự xuất hiện); trả tối đa `max_keep` lời giải,
  nguyên văn như lúc sinh.
- `word_ngrams(text, n) -> set[tuple[str, ...]]`: các n-gram từ của `text`, từ là `\w+` sau khi `lower()`.
- `contaminated(train_questions, eval_questions, n=13) -> set[int]`: chỉ số các câu train có chung **ít nhất một**
  n-gram từ với **bất kỳ** câu nào của tập đo (mức GPT-3 dùng cho kiểm nhiễm bẩn).
- `to_sft_record(question, trace, system) -> dict`: dạng prompt-completion hội thoại của TRL:
  `{"prompt": [system, user], "completion": [assistant]}`, mỗi phần tử là một dict `role`/`content`.

In [ ]:
SYSTEM_COT = "Please reason step by step, and put your final answer within \\boxed{}."


def select_traces(samples: list[str], gold: str, max_keep: int = 2) -> list[str]:
    kept, seen = [], set()
    for text in samples:
        if correctness_reward(text, gold) != 1.0 or format_reward(text) != 1.0:
            continue
        key = " ".join(text.split())
        if key in seen:
            continue
        seen.add(key)
        kept.append(text)
    return sorted(kept, key=len)[:max_keep]


def word_ngrams(text: str, n: int) -> set[tuple[str, ...]]:
    words = re.findall(r"\w+", text.lower())
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


def contaminated(train_questions: list[str], eval_questions: list[str], n: int = 13) -> set[int]:
    eval_grams = set().union(*(word_ngrams(q, n) for q in eval_questions)) if eval_questions else set()
    return {i for i, q in enumerate(train_questions) if word_ngrams(q, n) & eval_grams}


def to_sft_record(question: str, trace: str, system: str = SYSTEM_COT) -> dict:
    return {"prompt": [{"role": "system", "content": system}, {"role": "user", "content": question}],
            "completion": [{"role": "assistant", "content": trace}]}

## 3. Sinh lời giải trên tập train

Câu train: 800 câu đầu của tập train GSM8K trộn theo seed 0 (kernel CPU: 6 câu). Bỏ câu nhiễm bẩn với 500 câu test ghim,
rồi sinh 4 lời giải mỗi câu ở `temperature` 0.7. Cache ra đĩa. Khoảng 20 phút trên GPU của PC (đo ở `SOLUTION.md`).

In [ ]:
# explore: dữ liệu, kiểm nhiễm bẩn, sinh 4 lời giải mỗi câu
import json
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
N_SRC, K, MAX_NEW, BATCH, N_EVAL = (800, 4, 512, 128, 500) if GPU else (6, 2, 64, 8, 6)
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
train = pd.read_parquet(GSM / "main" / "train-00000-of-00001.parquet").sample(frac=1.0, random_state=0)
test = pd.read_parquet(GSM / "main" / "test-00000-of-00001.parquet").iloc[:500]
gold = lambda a: normalize_number(a.split("####")[-1])

for n in (8, 13):
    print(f"{n}-gram: {len(contaminated(list(train.question), list(test.question), n))} câu train trùng n-gram với 500 câu test")
src = train.iloc[:N_SRC]
drop = contaminated(list(src.question), list(test.question), 13)
src = src.iloc[[i for i in range(len(src)) if i not in drop]]
print("câu nguồn sau khi bỏ nhiễm bẩn:", len(src))

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
STOP = {tok.eos_token_id, tok.pad_token_id, tok.convert_tokens_to_ids("<|im_end|>")}


def chat_prompt(question):
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM_COT}, {"role": "user", "content": question}],
                                   tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def generate(model, questions, n, temperature, max_new, batch, seed=0):
    prompts = [chat_prompt(q) for q in questions for _ in range(n)]
    texts, lengths = [], []
    model.eval()
    for s in range(0, len(prompts), batch):
        torch.manual_seed(seed + s)
        enc = tok(prompts[s:s + batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        kw = (dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if temperature > 0
              else dict(do_sample=False, temperature=None, top_p=None, top_k=None))
        out = model.generate(**enc, max_new_tokens=max_new, pad_token_id=tok.pad_token_id, **kw)
        for row in out[:, enc["input_ids"].shape[1]:].tolist():
            end = next((j + 1 for j, t in enumerate(row) if t in STOP), len(row))
            texts.append(tok.decode(row[:end], skip_special_tokens=True))
            lengths.append(end)
    return texts, lengths


model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16 if GPU else torch.float32).to(DEVICE)
samples_file = CACHE / ("m15-rft-samples.json" if GPU else "m15-rft-samples-cpu.json")
if samples_file.exists():
    samples = json.loads(samples_file.read_text(encoding="utf-8"))
else:
    t0 = time.perf_counter()
    texts, lengths = generate(model, list(src.question), K, 0.7, MAX_NEW, BATCH)
    samples = {"texts": texts, "lengths": lengths, "seconds": round(time.perf_counter() - t0, 1)}
    samples_file.write_text(json.dumps(samples, ensure_ascii=False), encoding="utf-8")
print(f"{len(samples['texts'])} lời giải, {samples['seconds']} s")

In [ ]:
# explore: lọc, thống kê, dựng tập SFT
golds = [gold(a) for a in src.answer]
groups = [samples["texts"][i * K:(i + 1) * K] for i in range(len(src))]
kept = [select_traces(g, y, max_keep=2) for g, y in zip(groups, golds)]
correct_any = np.mean([any(correctness_reward(t, y) for t in g) for g, y in zip(groups, golds)])
records = [to_sft_record(q, t) for q, ks in zip(src.question, kept) for t in ks]
stats = {
    "lời giải đúng đáp số": float(np.mean([correctness_reward(t, y) for g, y in zip(groups, golds) for t in g])),
    "lời giải đúng cả khung": float(np.mean([correctness_reward(t, y) * format_reward(t) for g, y in zip(groups, golds) for t in g])),
    "câu có ít nhất một lời giải đúng (pass@4)": float(correct_any),
    "câu đóng góp mẫu SFT": int(sum(bool(k) for k in kept)),
    "số mẫu SFT": len(records),
    "độ dài TB mẫu giữ (ký tự)": float(np.mean([len(t) for k in kept for t in k])) if records else 0.0,
    "độ dài TB mọi lời giải (ký tự)": float(np.mean([len(t) for t in samples["texts"]])),
}
for k, v in stats.items():
    print(f"{k:45s} {v:.3f}" if isinstance(v, float) else f"{k:45s} {v}")
print("\nMột mẫu SFT:\n", records[0]["completion"][0]["content"][:600] if records else "(không có)")

## 4. SFT bằng LoRA rồi đo trước/sau

TRL `SFTTrainer` với dữ liệu prompt-completion: loss chỉ tính trên phần completion (module 14 mục 2). LoRA r = 16 trên mọi
lớp tuyến tính, 2 epoch, learning rate 1e-4. Đo greedy trên 500 câu test ghim (kernel CPU: 6 câu) với cùng prompt:
`accuracy` theo bộ trích dễ dãi, `boxed_correct` theo reward, `format`, độ dài.

In [ ]:
# explore: đo model gốc, SFT, đo lại, lưu kết quả
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def evaluate(model, name):
    t0 = time.perf_counter()
    texts, lengths = generate(model, list(test.question[:N_EVAL]), 1, 0.0, 512 if GPU else 64, 64 if GPU else 8)
    golds_eval = [gold(a) for a in test.answer[:N_EVAL]]
    res = {"accuracy": float(np.mean([extract_answer(t) == g for t, g in zip(texts, golds_eval)])),
           "boxed_correct": float(np.mean([correctness_reward(t, g) for t, g in zip(texts, golds_eval)])),
           "format": float(np.mean([format_reward(t) for t in texts])),
           "mean_tokens": float(np.mean(lengths)), "truncated": int(sum(l >= (512 if GPU else 64) for l in lengths)),
           "seconds": round(time.perf_counter() - t0, 1),
           "correct": [bool(extract_answer(t) == g) for t, g in zip(texts, golds_eval)]}
    print(name, {k: v for k, v in res.items() if k != "correct"})
    return res


results = {"stats": stats, "before": evaluate(model, "gốc")}
args = SFTConfig(output_dir=str(CACHE / "m15-rft-run"), num_train_epochs=2 if GPU else 1, learning_rate=1e-4,
                 per_device_train_batch_size=4, gradient_accumulation_steps=4, lr_scheduler_type="cosine",
                 warmup_steps=0.05, logging_steps=10, save_strategy="no", report_to="none", bf16=GPU,
                 max_length=1024, seed=0, use_cpu=not GPU)
trainer = SFTTrainer(model=model, args=args, train_dataset=Dataset.from_list(records), processing_class=tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear",
                                            task_type="CAUSAL_LM"))
t0 = time.perf_counter()
out = trainer.train()
results["train"] = {"seconds": round(time.perf_counter() - t0, 1), "loss": out.training_loss,
                    "log": [r for r in trainer.state.log_history if "loss" in r]}
tok.padding_side = "left"
results["after"] = evaluate(trainer.model, "sau SFT")
path = CACHE / ("m15-rft.json" if GPU else "m15-rft-cpu.json")
path.write_text(json.dumps(results, ensure_ascii=False, indent=1), encoding="utf-8")
print("đã ghi", path)

In [ ]:
# explore: bảng trước/sau và số câu đổi chiều (kernel CPU in thêm kết quả GPU đã cache nếu có)
def report(res):
    table = pd.DataFrame({k: {m: res[k][m] for m in ("accuracy", "boxed_correct", "format", "mean_tokens", "truncated")}
                          for k in ("before", "after")}).T
    display(table.round(3))
    a, b = np.array(res["before"]["correct"]), np.array(res["after"]["correct"])
    print(f"đúng -> sai: {int((a & ~b).sum())}, sai -> đúng: {int((~a & b).sum())} trên {len(a)} câu")


report(results)
gpu_cache = CACHE / "m15-rft.json"
if not GPU and gpu_cache.exists():
    print("Kết quả GPU đã cache:")
    report(json.loads(gpu_cache.read_text(encoding="utf-8")))

## Đọc kết quả

- Bao nhiêu phần trăm câu train có ít nhất một lời giải đúng trong 4 lần thử? Đó là trần dữ liệu mà model tự cấp được: câu
  khó hơn sức nó không bao giờ vào tập SFT. Điều đó nói gì về việc tự học mãi bằng cách này?
- 8-gram và 13-gram cho số câu "nhiễm bẩn" khác nhau thế nào? Đọc vài cặp trùng 8-gram: nhiễm bẩn thật hay chỉ là cụm từ
  chung của đề toán?
- Sau SFT, cột nào đổi nhiều nhất: đúng khung, độ dài, hay độ chính xác? Chênh lệch accuracy so với số câu đổi chiều
  hai phía có vượt nhiễu không?
- Lời giải giữ lại ngắn hơn trung bình mọi lời giải: chọn "ngắn trước" dạy model điều gì?

Sau đó làm phần trắc nghiệm của bài trong app.